# Phase 3: Build a Strong Retrieval Pipeline

## Step 7: Hybrid Search

### Learning

- Hybrid retrieval
- Candidate generation
- Score normalization
- Rank fusion
- Reciprocal Rank Fusion (RRF)
- Weighted retrieval
- Deduplication
- Retrieval recall

---

## Key Takeaways

- Lexical (BM25/Elasticsearch) and semantic (embeddings/Chroma) retrieval solve different problems and fail in different ways.
- Raw BM25 scores and cosine-similarity scores live on incompatible scales — adding them directly is misleading.
- **Reciprocal Rank Fusion (RRF)** combines *rank positions* instead of raw scores, which sidesteps the scale-mismatch problem.
- A chunk may be returned by only one retriever — the merge step must handle that gracefully.
- Hybrid search only improves the *candidate set*; it doesn't guarantee the best chunk is ranked first (that's what reranking, Step 8, is for).

---

## To do (mirrors the Roadmap 1:1)

1. Create a common retrieval result format
2. Run both retrievers for every query
3. Merge results by stable ID
4. Implement Reciprocal Rank Fusion
5. Avoid combining raw scores directly (normalize as a *separate* experiment)
6. Deduplicate similar results
7. Add retrieval-mode controls (vector / BM25 / hybrid-RRF)
8. Create a test query set
9. Experiment with weighted fusion
10. Send fused results to the model, with citations

## 0. Environment Setup — connect to both retrieval backends

Same connection + indexing pattern as Steps 4–6: Elasticsearch for lexical search,
Chroma for vector search. Nothing new conceptually here — just getting both
backends populated with the same document set so Step 7 can query them side by side.

In [8]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)

except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)

    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

{'name': '0e1f447b24d9', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'lHCCr2OyQ0yMLKjGmwdmBg', 'version': {'number': '8.19.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '93788a8c2882eb5b606510680fac214cff1c7a22', 'build_date': '2025-07-23T22:10:18.138212839Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [9]:
import time
import numpy as np
import chromadb
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma = chromadb.HttpClient(
    host="localhost",
    port=8000,
)

INDEX_NAME = "rag_documents_hybrid"
COLLECTION_NAME = "john_doe_profile_hybrid"

In [10]:
# Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]
print(f"Loaded {len(documents)} chunks.")

Loaded 22 chunks.


In [12]:
# Index into Elasticsearch (lexical side) — same pattern as Step 6
from elasticsearch.helpers import bulk

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "text": {"type": "text"}
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

actions = [
    {
        "_index": INDEX_NAME,
        "_id": str(doc["id"]),
        "_source": {"chunk_id": str(doc["id"]), "text": doc["text"]}
    }
    for doc in documents
]
success, failed = bulk(es, actions)
print("Successfully indexed into Elasticsearch:", success, "| Failed:", failed)

Successfully indexed into Elasticsearch: 22 | Failed: []


In [13]:
# Index into Chroma (semantic side) — same pattern as Steps 4/5
try:
    client_chroma.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME)

embedded_documents = []
for doc in documents:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=doc["text"])
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

collection.add(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

print(f"Indexed {collection.count()} documents into Chroma.")

Indexed 22 documents into Chroma.


## 1. Create a common retrieval result format

> Make Chroma and Elasticsearch return results in the same structure. This makes it
> easier to combine results from different systems.

Every retriever — regardless of backend — should return a list of dicts shaped like:

```python
{
    "chunk_id": "chunk-001",
    "text": "...",
    "source": "profile.txt",
    "rank": 1,
    "score": 0.87,
    "retriever": "vector"
}
```

`rank` is 1-based position within that retriever's own result list. `score` stays in
the retriever's *native* scale for now (BM25 relevance score, or cosine similarity) —
we deliberately do NOT normalize it here (see Section 5).

In [15]:
def make_result(chunk_id, text, source, rank, score, retriever):
    """Build one result dict in the common format shared by every retriever."""
    return {
        "chunk_id": str(chunk_id),
        "text": text,
        "source": source,
        "rank": rank,
        "score": score,
        "retriever": retriever,
    }

## 2. Run both retrievers for every query

> For each user query: (1) generate the query embedding, (2) retrieve the top ten
> vector results, (3) send the original query to Elasticsearch, (4) retrieve the top
> ten lexical results. Record the latency of each retrieval method separately.

Both functions below return results already shaped in the common format from
Section 1, and both time themselves independently so you can compare latency.

In [16]:
def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def vector_search(query_text, top_k=10):
    """Semantic retrieval via Chroma. Returns (results, latency_ms)."""
    t0 = time.perf_counter()

    query_embedding = get_embedding(query_text)
    raw = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "distances"]
    )

    ids = raw["ids"][0]
    texts = raw["documents"][0]
    distances = raw["distances"][0]

    results = [
        make_result(
            chunk_id=doc_id,
            text=text,
            source="profile.txt",
            rank=i + 1,
            score=1 / (1 + distance),  # distance -> similarity, native to this retriever
            retriever="vector",
        )
        for i, (doc_id, text, distance) in enumerate(zip(ids, texts, distances))
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


def lexical_search(query_text, top_k=10):
    """Lexical (BM25) retrieval via Elasticsearch. Returns (results, latency_ms)."""
    t0 = time.perf_counter()

    query = {
        "size": top_k,
        "query": {"match": {"text": query_text}}
    }
    raw = es.search(index=INDEX_NAME, body=query)

    results = [
        make_result(
            chunk_id=hit["_source"]["chunk_id"],
            text=hit["_source"]["text"],
            source="profile.txt",
            rank=i + 1,
            score=hit["_score"],
            retriever="bm25",
        )
        for i, hit in enumerate(raw["hits"]["hits"])
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


def run_both_retrievers(query_text, top_k=10):
    """Run vector + lexical retrieval for one query and report latency separately."""
    vector_results, vector_latency_ms = vector_search(query_text, top_k=top_k)
    lexical_results, lexical_latency_ms = lexical_search(query_text, top_k=top_k)

    print(f"Query: '{query_text}'")
    print(f"  vector retrieval:  {len(vector_results):>2} results in {vector_latency_ms:6.2f} ms")
    print(f"  lexical retrieval: {len(lexical_results):>2} results in {lexical_latency_ms:6.2f} ms")

    return vector_results, lexical_results


_ = run_both_retrievers("John innovation")

Query: 'John innovation'
  vector retrieval:  10 results in 1270.81 ms
  lexical retrieval: 10 results in  41.35 ms


## 3. Merge results by stable ID

> Use `chunk_id` to identify the same chunk across both systems. Create a merged
> result object containing vector rank, vector score, BM25 rank, BM25 score, source
> metadata, and original text. Handle chunks returned by only one retriever.

This is the first piece for you to implement. Build a dict keyed by `chunk_id` where
each entry tracks what each retriever saw for that chunk — defaulting the missing
side to `None` rather than skipping it.

In [19]:
def merge_by_chunk_id(vector_results, lexical_results):
    """Merge vector + lexical result lists into one dict keyed by chunk_id.

    Returns:
        {
            chunk_id: {
                "chunk_id": ...,
                "text": ...,
                "source": ...,
                "vector_rank": int | None,
                "vector_score": float | None,
                "bm25_rank": int | None,
                "bm25_score": float | None,
            },
            ...
        }
    """
    merged = {}

    # TODO: iterate vector_results.
    #   For each result, get-or-create merged[chunk_id] with the fields above
    #   (text/source only need to be set once), then fill in vector_rank/vector_score.

    # TODO: iterate lexical_results the same way, filling in bm25_rank/bm25_score.
    #   A chunk_id may already exist from the vector pass — don't overwrite text/source,
    #   just add the bm25_* fields.


    # -----------------------------
    # 1. Add vector search results
    # -----------------------------
    for rank, result in enumerate(vector_results, start=1):
        chunk_id = result["chunk_id"]

        # Create the entry if this chunk hasn't been seen yet
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result.get("text"),
                "source": result.get("source"),
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }

        # Fill vector-specific information
        merged[chunk_id]["vector_rank"] = rank
        merged[chunk_id]["vector_score"] = result.get("score")


    # -----------------------------
    # 2. Add BM25 / lexical results
    # -----------------------------
    for rank, result in enumerate(lexical_results, start=1):
        chunk_id = result["chunk_id"]

        # Create the entry if this chunk only exists
        # in lexical results
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result.get("text"),
                "source": result.get("source"),
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }

        # Fill BM25-specific information
        merged[chunk_id]["bm25_rank"] = rank
        merged[chunk_id]["bm25_score"] = result.get("score")


    return merged
    # raise NotImplementedError("Merge vector_results and lexical_results by chunk_id")



# Try it:
vector_results, lexical_results = run_both_retrievers("John innovation")
merged = merge_by_chunk_id(vector_results, lexical_results)
for chunk_id, entry in merged.items():
    print(
        chunk_id,
        "vector_rank =", entry["vector_rank"],
        "bm25_rank =", entry["bm25_rank"]
    )

Query: 'John innovation'
  vector retrieval:  10 results in 2541.24 ms
  lexical retrieval: 10 results in  26.74 ms
8 vector_rank = 1 bm25_rank = None
18 vector_rank = 2 bm25_rank = 3
15 vector_rank = 3 bm25_rank = 6
11 vector_rank = 4 bm25_rank = None
16 vector_rank = 5 bm25_rank = 9
9 vector_rank = 6 bm25_rank = None
21 vector_rank = 7 bm25_rank = 2
19 vector_rank = 8 bm25_rank = None
0 vector_rank = 9 bm25_rank = 1
2 vector_rank = 10 bm25_rank = 4
1 vector_rank = None bm25_rank = 5
20 vector_rank = None bm25_rank = 7
7 vector_rank = None bm25_rank = 8
3 vector_rank = None bm25_rank = 10


## 4. Implement Reciprocal Rank Fusion

> `RRF score = Σ 1 / (k + rank)`. For every result: (1) calculate its contribution
> from vector rank, (2) calculate its contribution from BM25 rank, (3) add both
> contributions, (4) sort by the final RRF score. Expose `k` as a configurable
> parameter.

RRF combines **rank positions**, not raw scores — that's what makes it robust to the
scale mismatch between cosine similarity and BM25 (see Section 5). A chunk missing
from one retriever simply contributes 0 for that side.

In [20]:
def reciprocal_rank_fusion(merged, k=60):
    """Compute RRF scores for a merged result dict (from Section 3).

    Args:
        merged: output of merge_by_chunk_id().
        k: RRF constant. 60 is the commonly used default.

    Returns:
        List of merged entries with an added "rrf_score" field,
        sorted descending by rrf_score.
    """
    
    results = []

    for chunk_id, entry in merged.items():

        # Vector contribution
        if entry["vector_rank"] is not None:
            vector_contribution = 1 / (k + entry["vector_rank"])
        else:
            vector_contribution = 0


        # BM25 contribution
        if entry["bm25_rank"] is not None:
            bm25_contribution = 1 / (k + entry["bm25_rank"])
        else:
            bm25_contribution = 0


        # Combined RRF score
        rrf_score = vector_contribution + bm25_contribution

        # Store the score
        entry["rrf_score"] = rrf_score

        # Add result to list
        results.append(entry)


    # Sort highest RRF score first
    results.sort(
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return results

In [21]:
rrf_results = reciprocal_rank_fusion(merged)

for result in rrf_results:
    print(
        result["chunk_id"],
        "vector_rank =", result["vector_rank"],
        "bm25_rank =", result["bm25_rank"],
        "rrf_score =", result["rrf_score"]
    )

18 vector_rank = 2 bm25_rank = 3 rrf_score = 0.03200204813108039
21 vector_rank = 7 bm25_rank = 2 rrf_score = 0.031054405392392875
15 vector_rank = 3 bm25_rank = 6 rrf_score = 0.031024531024531024
0 vector_rank = 9 bm25_rank = 1 rrf_score = 0.030886196246139225
2 vector_rank = 10 bm25_rank = 4 rrf_score = 0.029910714285714284
16 vector_rank = 5 bm25_rank = 9 rrf_score = 0.029877369007803793
8 vector_rank = 1 bm25_rank = None rrf_score = 0.01639344262295082
11 vector_rank = 4 bm25_rank = None rrf_score = 0.015625
1 vector_rank = None bm25_rank = 5 rrf_score = 0.015384615384615385
9 vector_rank = 6 bm25_rank = None rrf_score = 0.015151515151515152
20 vector_rank = None bm25_rank = 7 rrf_score = 0.014925373134328358
19 vector_rank = 8 bm25_rank = None rrf_score = 0.014705882352941176
7 vector_rank = None bm25_rank = 8 rrf_score = 0.014705882352941176
3 vector_rank = None bm25_rank = 10 rrf_score = 0.014285714285714285


## 5. Avoid combining raw scores directly

> Keep cosine similarity and BM25 scores separate. Their ranges and distributions
> are different, so directly adding them can produce misleading results. As a
> separate experiment, normalize scores and compare the results with rank fusion.

This section is intentionally **not** used inside `reciprocal_rank_fusion` above —
RRF never touches raw scores. `normalize_scores` exists purely so you can run the
alternative experiment: min-max normalize each retriever's scores to `[0, 1]`, then
combine with a weighted sum, and compare that ranking against RRF's ranking.

In [22]:
def normalize_scores(results):
    """Min-max normalize the 'score' field of a list of result dicts to [0, 1].

    Adds a 'normalized_score' field to each dict in place and returns the list.
    If all scores are identical (or the list is empty), normalized_score = 1.0
    to avoid divide-by-zero.
    """
    if not results:
        return results

    scores = [r["score"] for r in results]
    min_score, max_score = min(scores), max(scores)

    if max_score == min_score:
        for r in results:
            r["normalized_score"] = 1.0
        return results

    for r in results:
        r["normalized_score"] = (r["score"] - min_score) / (max_score - min_score)

    return results


# Experiment scaffold — compare normalized-score fusion vs RRF on the same query:
# vector_results, lexical_results = run_both_retrievers("John innovation")
# normalize_scores(vector_results)
# normalize_scores(lexical_results)
# TODO: build a normalized-score-weighted ranking the same way merge_by_chunk_id
#   does, but summing normalized_score instead of using rank, then compare the
#   resulting order against reciprocal_rank_fusion()'s order for the same query.

In [23]:
def normalized_score_fusion(
    vector_results,
    lexical_results,
    vector_weight=0.5,
    lexical_weight=0.5
):
    """
    Combine vector and lexical results using normalized scores.

    Args:
        vector_results: Results from vector search.
        lexical_results: Results from BM25 / Elasticsearch.
        vector_weight: Weight given to vector scores.
        lexical_weight: Weight given to lexical scores.

    Returns:
        Combined results sorted by fused_score descending.
    """

    # Normalize scores first
    normalize_scores(vector_results)
    normalize_scores(lexical_results)

    merged = {}

    # Add vector results
    for result in vector_results:
        chunk_id = result["chunk_id"]

        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "vector_score": None,
                "vector_normalized_score": 0.0,
                "bm25_score": None,
                "bm25_normalized_score": 0.0,
            }

        merged[chunk_id]["vector_score"] = result["score"]
        merged[chunk_id]["vector_normalized_score"] = (
            result["normalized_score"]
        )

    # Add lexical/BM25 results
    for result in lexical_results:
        chunk_id = result["chunk_id"]

        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "vector_score": None,
                "vector_normalized_score": 0.0,
                "bm25_score": None,
                "bm25_normalized_score": 0.0,
            }

        merged[chunk_id]["bm25_score"] = result["score"]
        merged[chunk_id]["bm25_normalized_score"] = (
            result["normalized_score"]
        )

    # Calculate weighted combined score
    results = []

    for entry in merged.values():

        fused_score = (
            vector_weight * entry["vector_normalized_score"]
            + lexical_weight * entry["bm25_normalized_score"]
        )

        entry["fused_score"] = fused_score

        results.append(entry)

    # Highest fused score first
    results.sort(
        key=lambda x: x["fused_score"],
        reverse=True
    )

    return results

In [24]:
vector_results, lexical_results = run_both_retrievers(
    "John innovation"
)

fused_results = normalized_score_fusion(
    vector_results,
    lexical_results
)

for result in fused_results:
    print(
        result["chunk_id"],
        "vector =", result["vector_normalized_score"],
        "bm25 =", result["bm25_normalized_score"],
        "fused =", result["fused_score"]
    )

Query: 'John innovation'
  vector retrieval:  10 results in 1457.69 ms
  lexical retrieval: 10 results in  39.90 ms
0 vector = 0.07480339779709677 bm25 = 1.0 fused = 0.5374016988985484
8 vector = 1.0 bm25 = 0.0 fused = 0.5
18 vector = 0.987277927828145 bm25 = 0.006566859721310957 fused = 0.496922393774728
21 vector = 0.16756413335828382 bm25 = 0.7297908990051933 fused = 0.44867751618173857
15 vector = 0.6553164075269117 bm25 = 0.0011054138767029463 fused = 0.3282109107018073
11 vector = 0.5601193057845189 bm25 = 0.0 fused = 0.2800596528922594
16 vector = 0.4912469554417015 bm25 = 0.0003563863437160748 fused = 0.2458016708927088
9 vector = 0.36940883095367527 bm25 = 0.0 fused = 0.18470441547683764
19 vector = 0.16558880920391278 bm25 = 0.0 fused = 0.08279440460195639
2 vector = 0.0 bm25 = 0.005425716161382819 fused = 0.0027128580806914096
1 vector = 0.0 bm25 = 0.003876717241974229 fused = 0.0019383586209871144
20 vector = 0.0 bm25 = 0.0011054138767029463 fused = 0.0005527069383514732
7 

In [25]:
# RRF
merged = merge_by_chunk_id(
    vector_results,
    lexical_results
)

rrf_results = reciprocal_rank_fusion(merged)

print("RRF ranking:")
for result in rrf_results:
    print(
        result["chunk_id"],
        result["rrf_score"]
    )

print("\nNormalized-score fusion ranking:")

for result in fused_results:
    print(
        result["chunk_id"],
        result["fused_score"]
    )

RRF ranking:
18 0.03200204813108039
21 0.031054405392392875
15 0.031024531024531024
0 0.030886196246139225
2 0.029910714285714284
16 0.029877369007803793
8 0.01639344262295082
11 0.015625
1 0.015384615384615385
9 0.015151515151515152
20 0.014925373134328358
19 0.014705882352941176
7 0.014705882352941176
3 0.014285714285714285

Normalized-score fusion ranking:
0 0.5374016988985484
8 0.5
18 0.496922393774728
21 0.44867751618173857
15 0.3282109107018073
11 0.2800596528922594
16 0.2458016708927088
9 0.18470441547683764
19 0.08279440460195639
2 0.0027128580806914096
1 0.0019383586209871144
20 0.0005527069383514732
7 0.0003623286020868659
3 0.0


## 6. Deduplicate similar results

> Two chunks may contain almost identical content because of chunk overlap,
> duplicate source documents, reindexed documents, or different versions. Start by
> deduplicating exact chunk IDs. Later, add text-hash or similarity-based
> deduplication.

The merge step in Section 3 already deduplicates exact `chunk_id`s (since it's a
dict keyed by `chunk_id`). What's left here is **content-level** deduplication —
different IDs that happen to hold near-identical text.

In [26]:
import hashlib

def content_hash(text):
    """Stable hash of normalized chunk text, for exact-duplicate detection."""
    normalized = " ".join(text.lower().split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def deduplicate_by_content_hash(fused_results):
    """Given RRF-sorted results, drop later duplicates that share a content hash
    with an earlier (higher-ranked) result, keeping the first (best-ranked) copy.
    """
    seen_hashes = set()
    deduped = []

    for result in fused_results:
        h = content_hash(result["text"])
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        deduped.append(result)

    return deduped


# TODO (optional, later): replace exact-hash matching with similarity-based
# dedup — e.g. compare embeddings of near-neighbor chunks and drop any pair
# above a cosine-similarity threshold (~0.97+).

## 7. Add retrieval-mode controls

> Allow the learner to choose: vector only, BM25 only, hybrid with RRF. Display the
> results side by side.

In [27]:
RETRIEVAL_MODE_VECTOR = "vector"
RETRIEVAL_MODE_BM25 = "bm25"
RETRIEVAL_MODE_HYBRID = "hybrid"


def hybrid_search(query_text, mode=RETRIEVAL_MODE_HYBRID, top_k=5, candidate_k=10, rrf_k=60):
    """Single entry point supporting all three retrieval modes.

    vector / bm25 -> just that retriever's top_k, in the common format.
    hybrid        -> run both, merge (Section 3), fuse with RRF (Section 4),
                      dedupe (Section 6), return top_k.
    """
    if mode == RETRIEVAL_MODE_VECTOR:
        results, _ = vector_search(query_text, top_k=top_k)
        return results

    elif mode == RETRIEVAL_MODE_BM25:
        results, _ = lexical_search(query_text, top_k=top_k)
        return results

    elif mode == RETRIEVAL_MODE_HYBRID:
        vector_results, lexical_results = run_both_retrievers(query_text, top_k=candidate_k)
        merged = merge_by_chunk_id(vector_results, lexical_results)
        fused = reciprocal_rank_fusion(merged, k=rrf_k)
        deduped = deduplicate_by_content_hash(fused)
        return deduped[:top_k]

    else:
        raise ValueError(f"Unknown retrieval mode: {mode}")


def compare_modes_side_by_side(query_text, top_k=5):
    """Run all three modes on the same query and print results side by side."""
    for mode in (RETRIEVAL_MODE_VECTOR, RETRIEVAL_MODE_BM25, RETRIEVAL_MODE_HYBRID):
        print(f"\n=== mode: {mode} ===")
        results = hybrid_search(query_text, mode=mode, top_k=top_k)
        for r in results:
            score_field = "rrf_score" if mode == RETRIEVAL_MODE_HYBRID else "score"
            print(f"  chunk={r['chunk_id']:>3}  {score_field}={r[score_field]:.4f}  {r['text'][:70]}...")

## 8. Create a test query set

> Include queries for: exact identifiers, semantic paraphrases, names, acronyms,
> policies, multi-word concepts, very short queries. Record which retriever
> performs better for each query.

In [28]:
test_queries = [
    # category: query
    ("exact_identifier", "EMP-78432"),
    ("semantic_paraphrase", "Where did he go to school?"),
    ("name", "John Doe"),
    ("multi_word_concept", "lifelong learning and mentorship"),
    ("very_short", "innovation"),
]

for category, query in test_queries:
    print(f"\n########## [{category}] \"{query}\" ##########")
    compare_modes_side_by_side(query, top_k=3)

# TODO: after running this, note down (in a markdown cell or comments) which
# retriever won for each category — this observation directly feeds Section 9.


########## [exact_identifier] "EMP-78432" ##########

=== mode: vector ===
  chunk=  8  score=0.3699  Upon completing his degree in 2009, John accepted a position as a juni...
  chunk= 12  score=0.3619  In 2018, John co-founded a fictional startup called BrightPath Technol...
  chunk=  9  score=0.3613  Over the next several years, John gained expertise in full-stack softw...

=== mode: bm25 ===

=== mode: hybrid ===
Query: 'EMP-78432'
  vector retrieval:  10 results in 451.75 ms
  lexical retrieval:  0 results in  11.24 ms
  chunk=  8  rrf_score=0.0164  Upon completing his degree in 2009, John accepted a position as a juni...
  chunk= 12  rrf_score=0.0161  In 2018, John co-founded a fictional startup called BrightPath Technol...
  chunk=  9  rrf_score=0.0159  Over the next several years, John gained expertise in full-stack softw...

########## [semantic_paraphrase] "Where did he go to school?" ##########

=== mode: vector ===
  chunk=  5  score=0.4472  After graduating with honors, Jo

## 9. Experiment with weighted fusion

> Add optional weights: vector weight, lexical weight. For example, give lexical
> retrieval more influence when the query contains numbers, hyphens, capitalized
> codes, or known identifier formats.

This extends RRF with per-retriever weights:
`weighted_rrf_score = vector_weight * (1/(k+vector_rank)) + lexical_weight * (1/(k+bm25_rank))`.

In [29]:
import re

def looks_like_identifier(query_text):
    """Heuristic: does this query look like an exact identifier rather than
    a natural-language question? (numbers, hyphens, capitalized codes, etc.)
    """
    return bool(re.search(r"[0-9]", query_text)) or bool(re.search(r"-", query_text)) \
        or query_text.isupper()


def weighted_rank_fusion(merged, k=60, vector_weight=0.5, lexical_weight=0.5):
    """Same idea as reciprocal_rank_fusion, but each retriever's contribution
    is scaled by a configurable weight before summing.
    """
    results = []

    # TODO: same loop as Section 4's reciprocal_rank_fusion, but multiply
    #   the vector contribution by vector_weight and the bm25 contribution
    #   by lexical_weight before summing into "weighted_score".

    raise NotImplementedError("Implement weighted rank fusion")


def adaptive_hybrid_search(query_text, top_k=5, candidate_k=10, k=60):
    """Automatically favor lexical retrieval for identifier-like queries."""
    if looks_like_identifier(query_text):
        vector_weight, lexical_weight = 0.2, 0.8
    else:
        vector_weight, lexical_weight = 0.5, 0.5

    vector_results, lexical_results = run_both_retrievers(query_text, top_k=candidate_k)
    merged = merge_by_chunk_id(vector_results, lexical_results)
    fused = weighted_rank_fusion(merged, k=k, vector_weight=vector_weight, lexical_weight=lexical_weight)
    deduped = deduplicate_by_content_hash(fused)
    return deduped[:top_k]

## 10. Send fused results to the model

> Select the top five fused chunks. Format them with stable source numbers.
> Ask the model to answer using only the retrieved material.

In [30]:
def format_sources(results):
    """Format fused results with stable [n] source numbers for the prompt."""
    lines = []
    for i, r in enumerate(results, start=1):
        lines.append(f"[{i}] {r['text']}")
    return "\n\n".join(lines)


def answer_with_hybrid_search(user_question, top_k=5):
    results = hybrid_search(user_question, mode=RETRIEVAL_MODE_HYBRID, top_k=top_k)
    sources_block = format_sources(results)

    system_prompt = f"""You answer questions using ONLY the retrieved sources below.
Cite claims using the matching [n] source number.
If the sources do not contain the answer, say so plainly — do not guess.

Retrieved sources:
{sources_block}"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )
    return response.choices[0].message.content, results


answer, sources = answer_with_hybrid_search("What did John study and where?")
print(answer)

Query: 'What did John study and where?'
  vector retrieval:  10 results in 510.06 ms
  lexical retrieval: 10 results in   6.95 ms
John studied Computer Science at the fictional North Valley University, where he pursued a Bachelor of Science degree [2].


## Step 8: Reranking

### Learning

- First-stage retrieval
- Candidate generation
- Second-stage ranking
- Cross-encoders
- Pairwise relevance
- Recall versus precision
- Reranking latency
- Candidate-set size


## 1. Increase first-stage retrieval depth

> Retrieve a broader candidate set: 10 vector results, 10 lexical results, 20 or
> fewer fused candidates. The first stage should prioritize finding potentially
> relevant information rather than perfectly ordering it.

Step 7's `hybrid_search()` already accepts `candidate_k` (per-retriever depth) and
`top_k` (how many fused results to return). For reranking we want a *wider* funnel
than we'd hand straight to the generation model — pull more candidates through the
fusion stage, then let the reranker narrow it back down.

In [31]:
# Widen the first-stage funnel: 10 vector + 10 lexical candidates,
# fused down to at most 20 chunks (instead of Step 7's tighter top_k=5).
RERANK_CANDIDATE_K = 10   # per-retriever depth fed into fusion
RERANK_FUSED_K = 20       # how many fused candidates survive into reranking

def get_rerank_candidates(query_text):
    """Run Step 7's hybrid search wide enough to give the reranker real choices."""
    return hybrid_search(
        query_text,
        mode=RETRIEVAL_MODE_HYBRID,
        top_k=RERANK_FUSED_K,
        candidate_k=RERANK_CANDIDATE_K,
    )


candidates = get_rerank_candidates("What did John study and where?")
print(f"{len(candidates)} candidates ready for reranking")

Query: 'What did John study and where?'
  vector retrieval:  10 results in 1957.04 ms
  lexical retrieval: 10 results in  27.09 ms
14 candidates ready for reranking


## 2. Create a reranking interface

> Define `rerank(query, documents) -> list[dict]`. It should accept the original
> user query and candidate chunks, and return the same chunks with a reranker score
> and reranker rank added.

This is the shared contract every reranker implementation (LLM-based, cross-encoder,
whatever comes next) will follow, so the rest of the pipeline doesn't care which one
is plugged in.

In [32]:
def rerank(query: str, documents: list[dict]) -> list[dict]:
    """Interface every reranker implementation should satisfy.

    Args:
        query: the original user query (not a rewritten/standalone version).
        documents: candidate chunks from Step 7 (common result format).

    Returns:
        The same chunks, each with two added fields:
            "reranker_score": float
            "reranker_rank":  int (1-based, 1 = most relevant)
        sorted descending by reranker_score.
    """
    raise NotImplementedError(
        "Placeholder — Section 3 implements this via an LLM reranker, "
        "call llm_rerank(query, documents) directly for now."
    )

## 3. Build an LLM-based reranker first

> Before using a dedicated reranking model, ask an LLM to score each query-document
> pair. Require structured output: `{"relevance_score": 0, "reason": "..."}`. Use a
> fixed scale such as 0–10. The prompt should ask whether the chunk contains
> information that *helps answer* the query, not whether the chunk is *generally
> related*.

Note the distinction the roadmap draws: "helps answer" vs. "generally related" is
exactly the gap between a reranker and a retriever. A chunk can be topically similar
(high embedding similarity) without containing the actual answer.

In [33]:
import json as _json
from pydantic import BaseModel, Field


class RelevanceScore(BaseModel):
    relevance_score: int = Field(ge=0, le=10)
    reason: str


def score_single_pair(query: str, chunk_text: str) -> RelevanceScore:
    """Ask the LLM to score ONE query-document pair on a 0-10 scale.

    TODO:
      1. Build a prompt that gives the model the query and the chunk text, and
         asks: does this chunk contain information that helps ANSWER the query
         (not just "is it related to the topic")?
      2. Call client.chat.completions.create(...) using MODEL_NAME, requesting
         structured output that matches the RelevanceScore schema (0-10 score
         + a short reason).
      3. Parse and validate the response with RelevanceScore(**parsed_json).
      4. Return the validated RelevanceScore.
    """
    raise NotImplementedError("Implement single-pair LLM scoring")


def llm_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Naive LLM reranker: one model call per candidate (see Section 4 for batching)."""
    scored = []
    for doc in documents:
        result = score_single_pair(query, doc["text"])
        doc = {**doc, "reranker_score": result.relevance_score, "reason": result.reason}
        scored.append(doc)

    scored.sort(key=lambda d: d["reranker_score"], reverse=True)
    for i, doc in enumerate(scored, start=1):
        doc["reranker_rank"] = i

    return scored

## 4. Batch candidates where practical

> Instead of making one model request per candidate, provide several candidates in
> a single request. Require the model to return a score for each stable chunk ID.
> Validate that every chunk received a score, no unknown chunk IDs were created,
> and scores are within the allowed range.

This replaces Section 3's one-call-per-chunk approach with a single call that scores
the whole candidate set — much cheaper and faster for `RERANK_FUSED_K` candidates.

In [34]:
class ChunkScore(BaseModel):
    chunk_id: str
    relevance_score: int = Field(ge=0, le=10)


class BatchRerankResult(BaseModel):
    scores: list[ChunkScore]


def batch_llm_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Score all candidates in a single model call, then validate the response.

    TODO:
      1. Build a prompt listing every candidate as `[chunk_id] text`, and ask the
         model to return a 0-10 relevance score PER chunk_id, matching the
         BatchRerankResult schema.
      2. Call the model, requesting structured output.
      3. Parse into BatchRerankResult.
      4. Validate:
           - every input chunk_id received a score (no missing chunks)
           - no unknown chunk_id appears in the response that wasn't in `documents`
           - every score is within [0, 10] (Pydantic's Field already enforces this,
             but confirm the count matches len(documents))
         If validation fails, decide: retry, fall back to score 0, or raise.
      5. Attach reranker_score to each document, sort descending, assign
         reranker_rank, and return — same output contract as llm_rerank().
    """
    raise NotImplementedError("Implement batched LLM reranking with validation")

## 5. Add a dedicated reranking model

> After understanding the logic, connect a dedicated reranking model or
> cross-encoder. Send query + candidate texts, retrieve relevance scores. Compare
> its latency and ranking with the LLM-based method.

A cross-encoder (e.g. a `sentence-transformers` CrossEncoder model, or a hosted
reranking API like Cohere Rerank) scores query-document pairs directly, without
going through a full chat completion — typically much faster and cheaper per
candidate than an LLM call.

In [35]:
# Optional dependency — install with: pip install sentence-transformers --break-system-packages
#
# from sentence_transformers import CrossEncoder
# cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Rerank using a dedicated cross-encoder model instead of an LLM.

    TODO:
      1. Build (query, chunk_text) pairs for every candidate.
      2. cross_encoder.predict(pairs) -> raw relevance scores (float array).
      3. Attach reranker_score to each document, sort descending, assign
         reranker_rank — same output contract as llm_rerank() / batch_llm_rerank().
      4. Time this function the same way as Section 9 and compare against the
         LLM-based reranker on the same candidate set.
    """
    raise NotImplementedError("Wire up a cross-encoder (or hosted rerank API) and compare")

## 6. Keep only the best chunks

> After reranking, pass only the top three to five chunks to the generation model.
> Record: candidate count before reranking, candidate count after reranking,
> reranking duration, final context token count.

In [36]:
import tiktoken

def finalize_context(query: str, reranked_documents: list[dict], keep_top_n=5):
    """Trim reranked candidates down to the final generation context and report stats."""
    final_chunks = reranked_documents[:keep_top_n]

    encoding = tiktoken.encoding_for_model("gpt-4o")  # any cl100k/o200k-family encoder is fine for counting
    final_context_tokens = sum(len(encoding.encode(c["text"])) for c in final_chunks)

    stats = {
        "candidates_before_rerank": len(reranked_documents),
        "candidates_after_rerank": len(final_chunks),
        "final_context_tokens": final_context_tokens,
    }
    return final_chunks, stats


# candidates = get_rerank_candidates("What did John study and where?")
# reranked = batch_llm_rerank("What did John study and where?", candidates)
# final_chunks, stats = finalize_context("...", reranked, keep_top_n=5)
# print(stats)

## 7. Show before-and-after ranking

> Display a table with: chunk ID, vector rank, BM25 rank, RRF rank, reranker rank,
> reranker score. Identify chunks that moved significantly.

This stitches together every rank Step 7 + Step 8 produced for the same chunk, so
you can see exactly what reranking changed versus first-stage fusion.

In [37]:
def show_before_after(query_text, keep_top_n=5, move_threshold=3):
    """Print a before/after ranking table for one query.

    TODO:
      1. Get first-stage fused candidates AND their pre-rerank order:
           - re-run merge_by_chunk_id + reciprocal_rank_fusion (Step 7) to capture
             vector_rank / bm25_rank / rrf-implied rank per chunk_id
           - or extend get_rerank_candidates() to also return the merged dict
      2. Assign an "rrf_rank" = 1-based position in the RRF-sorted candidate list.
      3. Run batch_llm_rerank() (or cross_encoder_rerank()) on those candidates to
         get reranker_score / reranker_rank.
      4. Print one row per chunk: chunk_id, vector_rank, bm25_rank, rrf_rank,
         reranker_rank, reranker_score.
      5. Flag rows where abs(rrf_rank - reranker_rank) >= move_threshold as
         "moved significantly".
    """
    raise NotImplementedError("Build the combined before/after ranking table")

## 8. Create difficult retrieval examples

> Test cases where: several chunks discuss the same broad topic, only one chunk
> directly answers the question, a high-similarity chunk lacks the answer, an exact
> keyword match is misleading, the answer contains a negation or exception.

In [38]:
difficult_queries = [
    # category: query — fill in / adapt to your profile.txt content
    ("same_broad_topic", "Tell me about John's career."),               # many chunks mention career
    ("one_chunk_has_answer", "What is John's employee ID?"),            # only one chunk has the exact fact
    ("high_similarity_no_answer", "What programming language does John prefer?"),
    ("misleading_keyword_match", "John Smith"),                         # exact-keyword trap if no John Smith exists
    ("negation_or_exception", "When is John NOT available for meetings?"),
]

# TODO: for each (category, query), run get_rerank_candidates() then
# batch_llm_rerank(), and manually inspect whether the top reranked chunk
# actually contains the answer. Note failures — these become Step 9's
# regression/eval fodder later in the roadmap.

## 9. Measure the latency trade-off

> Track: embedding time, vector-search time, Elasticsearch time, fusion time,
> reranking time, generation time, total response time. Determine whether
> reranking should run for every query.

In [40]:
def timed_end_to_end(query_text, keep_top_n=5):
    """Run the full Step 7 + Step 8 pipeline once, timing every stage separately."""
    timings = {}

    t0 = time.perf_counter()
    vector_results, lexical_results = run_both_retrievers(query_text, top_k=RERANK_CANDIDATE_K)
    timings["retrieval_total_ms"] = (time.perf_counter() - t0) * 1000
    # NOTE: vector_search()/lexical_search() already report their own latency
    # individually if you want a finer breakdown than the combined number above.

    t1 = time.perf_counter()
    merged = merge_by_chunk_id(vector_results, lexical_results)
    fused = reciprocal_rank_fusion(merged, k=60)
    deduped = deduplicate_by_content_hash(fused)[:RERANK_FUSED_K]
    timings["fusion_ms"] = (time.perf_counter() - t1) * 1000

    t2 = time.perf_counter()
    reranked = batch_llm_rerank(query_text, deduped)
    timings["reranking_ms"] = (time.perf_counter() - t2) * 1000

    t3 = time.perf_counter()
    final_chunks, stats = finalize_context(query_text, reranked, keep_top_n=keep_top_n)
    sources_block = format_sources(final_chunks)
    system_prompt = f"""You answer questions using ONLY the retrieved sources below.
Cite claims using the matching [n] source number.
If the sources do not contain the answer, say so plainly.

Retrieved sources:
{sources_block}"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query_text},
        ],
    )
    timings["generation_ms"] = (time.perf_counter() - t3) * 1000

    timings["total_ms"] = sum(timings.values())
    return response.choices[0].message.content, timings


# answer, timings = timed_end_to_end("What did John study and where?")
# print(timings)
# print(answer)

## 10. Add a reranking threshold

> If no candidate receives a sufficiently strong reranker score, tell the
> generation layer that the evidence may be insufficient. Do not force the model
> to answer from weak documents.

In [41]:
RERANK_MIN_SCORE = 5  # on the 0-10 scale used by score_single_pair / batch_llm_rerank

def answer_with_rerank(user_question, keep_top_n=5, min_score=RERANK_MIN_SCORE):
    """Full pipeline: hybrid retrieve -> rerank -> threshold check -> generate.

    TODO:
      1. candidates = get_rerank_candidates(user_question)
      2. reranked = batch_llm_rerank(user_question, candidates)
      3. If reranked is empty OR reranked[0]["reranker_score"] < min_score:
           - do NOT call the generation model with weak evidence
           - return a clear "insufficient evidence" response instead, e.g.
             "I couldn't find strong enough evidence in the documents to answer this."
      4. Otherwise, finalize_context() -> format_sources() -> generate the answer
         the same way Section 9's timed_end_to_end() does, and return it.
    """
    raise NotImplementedError("Implement the threshold check before generation")